# CLSA glaucoma: vessel, optic-disc, and foveal explainability

This notebook extends notebook 08 **only within CLSA**. It intersects
participant-held-out RETFound glaucoma contribution maps with retinal
anatomy generated by the MIT-licensed Berens Lab
[`fundus_image_toolbox`](https://github.com/berenslab/fundus_image_toolbox).

The package provides:

- an FR-U-Net ensemble vessel **segmentation** trained on FIVES; and
- EfficientNet fovea and optic-disc center **localization** trained on
  ADAM, REFUGE, and IDRID.

The optic-disc, peripapillary, and foveal outputs below are therefore
prespecified circular ROIs around localized centers—not pixel-level disc
or fovea segmentations. Every result preserves this distinction.

Package source is pinned to commit
`d7757e28fbf639856b53cfe00019f605af8c1f17` for reproducibility.


In [ ]:
%pip install -q "numpy>=2.0,<2.3" "git+https://github.com/berenslab/fundus_image_toolbox.git@d7757e28fbf639856b53cfe00019f605af8c1f17" "huggingface_hub>=0.24"


In [ ]:
%pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless


In [ ]:
%pip install -q --no-deps "opencv-python-headless==4.11.0.86"


In [ ]:
dbutils.library.restartPython()


In [ ]:
from pathlib import Path
import gc
import hashlib
import importlib
import json
import math
import os
import shutil
import sys
import time
import uuid

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy import ndimage

cv2.setNumThreads(1)
print("OpenCV headless smoke test:", cv2.__version__)
if cv2.__version__ != "4.11.0":
    raise RuntimeError(
        "Expected the Databricks-safe OpenCV 4.11.0 build, but imported "
        f"{cv2.__version__}. Restart Python and rerun from the top."
    )


In [ ]:
# All reproducible paths and analysis parameters are fixed below. The only
# intentionally temporary input is the gated-model credential, and it is not
# written to the notebook or repository.
try:
    dbutils.widgets.get("hf_token")
except Exception:
    dbutils.widgets.text("hf_token", "", "Hugging Face token (temporary)")


In [ ]:
# This cell is deliberately self-contained so it is safe to run immediately
# after dbutils.library.restartPython() or after a kernel recovery.
from pathlib import Path
import gc
import hashlib
import importlib
import json
import math
import os
import shutil
import sys
import time
import uuid

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy import ndimage

repo_root = Path(
    "/Workspace/Users/ad0038@pennmedicine.upenn.edu/CLSA/CLSA_retina"
)
age_glaucoma_root = Path(
    "/Volumes/ophthalmology_analytics/dev_optic/clsa_dataset/derived/"
    "clsa_retinal_aging/Age_Glaucoma"
)
notebook08_root = (
    age_glaucoma_root / "13_glaucoma_classifier_spatial_validation"
)
output_root = age_glaucoma_root / "14_clsa_anatomic_explainability"
fit_cache_dir = (
    age_glaucoma_root.parent
    / "model_checkpoints"
    / "fundus_image_toolbox"
)

segmentation_batch_size = 4
checkpoint_every_batches = 25
max_new_batches_per_run = 100
maximum_images = 0
resume_segmentation = True
vessel_threshold = 0.5
optic_disc_radius_scale = 0.20
fovea_radius_scale = 0.20
peripapillary_multiplier = 2.0
vessel_dilation_px = 2
permutations = 5000
bootstrap_repetitions = 2000
run_targeted_occlusion = False
occlusion_controls = 5
retfound_repo = None
checkpoint_path = None
allow_repo_clone = True
allow_downloads = True
device_requested = "auto"

if (
    segmentation_batch_size < 1
    or checkpoint_every_batches < 1
    or max_new_batches_per_run < 0
    or maximum_images < 0
):
    raise ValueError(
        "Batch/checkpoint sizes must be positive; run and image limits must "
        "be nonnegative"
    )
if not 0 < vessel_threshold < 1:
    raise ValueError("vessel_threshold must lie in (0, 1)")
if not 0 < optic_disc_radius_scale <= 0.5:
    raise ValueError("optic_disc_radius_scale must lie in (0, 0.5]")
if not 0 < fovea_radius_scale <= 0.5:
    raise ValueError("fovea_radius_scale must lie in (0, 0.5]")
if peripapillary_multiplier <= 1:
    raise ValueError("peripapillary_multiplier must exceed 1")
if vessel_dilation_px < 0 or occlusion_controls < 1:
    raise ValueError("Dilation cannot be negative; controls must be positive")
if permutations < 500 or bootstrap_repetitions < 500:
    raise ValueError("Use at least 500 permutations and bootstrap repetitions")

module_root = repo_root / "src"
if str(module_root) not in sys.path:
    sys.path.insert(0, str(module_root))

import age_gap_extremes as _age_gap_extremes  # noqa: E402
import clsa_anatomic_explainability as _anatomic  # noqa: E402
import fundus_retfound_pipeline as _fundus  # noqa: E402
import glaucoma_classifier_spatial as _glaucoma  # noqa: E402

_age_gap_extremes = importlib.reload(_age_gap_extremes)
_anatomic = importlib.reload(_anatomic)
_fundus = importlib.reload(_fundus)
_glaucoma = importlib.reload(_glaucoma)

from age_gap_extremes import fundus_physiology_proxies  # noqa: E402
from clsa_anatomic_explainability import (  # noqa: E402
    attribution_region_metrics,
    build_anatomic_masks,
    participant_permutation_inference,
    sample_translated_control_masks,
)
from fundus_retfound_pipeline import (  # noqa: E402
    QualityConfig,
    RETFoundConfig,
    linear_head_score_from_array,
    load_retfound_model,
    prepare_model_input,
    preprocess_fundus,
    write_frame,
    write_json,
)
from glaucoma_classifier_spatial import (  # noqa: E402
    sample_equal_area_control_masks,
)


In [ ]:
FIT_SOURCE_COMMIT = "d7757e28fbf639856b53cfe00019f605af8c1f17"
os.environ["FIT_CACHE_DIR"] = str(fit_cache_dir)
fit_cache_dir.mkdir(parents=True, exist_ok=True)
torch_cache_dir = fit_cache_dir / "torch"
torch_cache_dir.mkdir(parents=True, exist_ok=True)
os.environ["TORCH_HOME"] = str(torch_cache_dir)

import fundus_image_toolbox as fit  # noqa: E402

if device_requested == "auto":
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
elif device_requested == "cuda":
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is unavailable")
    device = "cuda:0"
else:
    device = "cpu"
print("Fundus Image Toolbox version:", fit.__version__)
print("Pinned source commit:", FIT_SOURCE_COMMIT)
print("Anatomy device:", device)
print("Persistent FIT cache:", fit_cache_dir)


## 1. Load the completed CLSA-only notebook 08 explanations


In [ ]:
attribution_manifest_path = (
    notebook08_root
    / "03_patch_attributions"
    / "glaucoma_attribution_manifest_private.parquet"
)
attribution_batch_root = notebook08_root / "03_patch_attributions" / "batches"
fold_heads_path = (
    notebook08_root
    / "01_participant_classifier"
    / "CLSA_glaucoma_oof_fold_heads.joblib"
)
required_paths = {
    "notebook 08 attribution manifest": attribution_manifest_path,
    "notebook 08 attribution batches": attribution_batch_root,
}
if run_targeted_occlusion:
    required_paths["notebook 08 fold heads"] = fold_heads_path
missing = [f"{name}: {path}" for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError("Required notebook 08 outputs are missing:\n- " + "\n- ".join(missing))

attribution_manifest = pd.read_parquet(attribution_manifest_path)
required_columns = {
    "image_key",
    "image_path",
    "participant_id",
    "source",
    "glaucoma_label",
    "fold",
    "classifier_logit_oof",
}
missing_columns = required_columns - set(attribution_manifest.columns)
if missing_columns:
    raise ValueError(
        "Notebook 08 attribution manifest is missing: "
        f"{sorted(missing_columns)}"
    )
clsa_images = attribution_manifest[
    attribution_manifest["source"].astype(str) == "CLSA"
].copy()
clsa_images["participant_id"] = clsa_images["participant_id"].astype(str)
clsa_images = clsa_images.drop_duplicates("image_path").sort_values(
    ["glaucoma_label", "participant_id", "image_path"], kind="stable"
)
if maximum_images:
    per_group = max(1, maximum_images // 2)
    stratified = [
        group.head(per_group)
        for _, group in clsa_images.groupby("glaucoma_label", sort=True)
    ]
    clsa_images = pd.concat(stratified, ignore_index=True).head(maximum_images)
if clsa_images.empty or set(clsa_images["glaucoma_label"].astype(int)) != {0, 1}:
    raise ValueError("CLSA anatomy analysis requires both glaucoma and healthy images")

map_paths = {}
for path in attribution_batch_root.rglob("*_exact_glaucoma_attribution.npz"):
    key = path.name.replace("_exact_glaucoma_attribution.npz", "")
    map_paths[key] = path
clsa_images["attribution_npz_path"] = clsa_images["image_key"].map(
    lambda key: str(map_paths.get(str(key), ""))
)
missing_maps = clsa_images["attribution_npz_path"].eq("")
if missing_maps.any():
    raise FileNotFoundError(
        f"{int(missing_maps.sum())} CLSA images lack saved exact attribution maps"
    )
print("CLSA images selected:", len(clsa_images))
print("CLSA participants:", clsa_images["participant_id"].nunique())
display(
    clsa_images.groupby("glaucoma_label").agg(
        images=("image_path", "count"),
        participants=("participant_id", "nunique"),
    )
)


## 2. Load pretrained vessel and landmark models

Weights are downloaded by the toolbox on first use and retained under
the governed `fit_cache_dir`. No participant image is sent externally.


In [ ]:
model_load_started = time.perf_counter()
print("[models 1/2] loading fovea/optic-disc localization model...", flush=True)
landmark_model, landmark_checkpoint = fit.load_fovea_od_model(
    device=device,
    cache_dir=str(fit_cache_dir),
)
print(
    "[models 1/2] localization model ready in "
    f"{time.perf_counter() - model_load_started:.1f}s",
    flush=True,
)
model_load_started = time.perf_counter()
print("[models 2/2] loading vessel-segmentation ensemble...", flush=True)
vessel_ensemble = fit.load_segmentation_ensemble(
    device=device,
    cache_dir=str(fit_cache_dir),
)
print(
    "[models 2/2] vessel ensemble ready in "
    f"{time.perf_counter() - model_load_started:.1f}s",
    flush=True,
)
print("Fovea/optic-disc checkpoint:", landmark_checkpoint)
print("Vessel ensemble models:", len(vessel_ensemble))


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


weight_paths = sorted(
    {
        path
        for pattern in ("*.pt", "*.pth", "*.ckpt")
        for path in fit_cache_dir.rglob(pattern)
        if path.is_file()
    }
)
weight_inventory = [
    {
        "filename": path.name,
        "relative_path": str(path.relative_to(fit_cache_dir)),
        "bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    }
    for path in weight_paths
]
write_json(
    {"models": weight_inventory},
    output_root / "FIT_MODEL_WEIGHT_INVENTORY.json",
)
print("Cached anatomy model files inventoried:", len(weight_inventory))


## 3. Optional RETFound targeted occlusion

The default run computes anatomy-specific attribution without replaying
RETFound. Set `run_targeted_occlusion=true` for the slower confirmatory
analysis. Start with `maximum_images=40` before scaling.


In [ ]:
quality_config = QualityConfig(
    output_size=256,
    model_input_size=224,
    save_preprocessed=False,
)
fold_heads = None
retfound_model = None
resolved_checkpoint = None
if run_targeted_occlusion:
    fold_heads = joblib.load(fold_heads_path)
    retfound_config = RETFoundConfig(
        repo_path=retfound_repo,
        checkpoint_path=checkpoint_path,
        allow_downloads=allow_downloads,
        allow_repo_clone=allow_repo_clone,
        device="cuda" if device.startswith("cuda") else "cpu",
        batch_size=1,
    )
    temporary_hf_token = dbutils.widgets.get("hf_token").strip()
    if allow_downloads and not checkpoint_path and not temporary_hf_token:
        raise ValueError(
            "Enter the temporary hf_token or provide checkpoint_path for "
            "targeted occlusion."
        )
    if temporary_hf_token:
        os.environ["HF_TOKEN"] = temporary_hf_token
    try:
        retfound_model, retfound_device, resolved_repo, resolved_checkpoint = (
            load_retfound_model(retfound_config)
        )
    finally:
        os.environ.pop("HF_TOKEN", None)
        temporary_hf_token = ""
    print("RETFound occlusion device:", retfound_device)
    print("RETFound checkpoint:", resolved_checkpoint)


In [ ]:
segmentation_root = output_root / "01_image_anatomy"
batch_root = segmentation_root / "batches"
checkpoint_root = segmentation_root / "checkpoints"
mask_root = segmentation_root / "masks"
overlay_root = output_root / "02_overlays_private"
statistics_root = output_root / "03_statistics"
figure_root = output_root / "04_figures"
for path in (
    batch_root,
    checkpoint_root,
    mask_root,
    overlay_root,
    statistics_root,
    figure_root,
):
    path.mkdir(parents=True, exist_ok=True)
local_artifact_root = Path(
    f"/local_disk0/tmp/clsa_anatomic_explainability_{os.getpid()}"
)
local_artifact_root.mkdir(parents=True, exist_ok=True)
print("Local anatomy artifact staging:", local_artifact_root)


def resize_mask(mask, shape):
    nearest = getattr(Image.Resampling, "NEAREST", Image.NEAREST)
    image = Image.fromarray(np.asarray(mask, dtype=np.uint8) * 255)
    return np.asarray(image.resize((shape[1], shape[0]), nearest)) > 0


def stable_key(value):
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()[:20]


def artifact_digest(path, chunk_bytes=8 * 1024 * 1024):
    """Return byte count and SHA-256 using ordinary Volume file access."""
    total_bytes = 0
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_bytes)
            if not chunk:
                break
            total_bytes += len(chunk)
            digest.update(chunk)
    return total_bytes, digest.hexdigest()


def publish_local_artifact(local_path, volume_path, retries=4):
    """Publish through the UC Volume FUSE path without ANY FILE privileges."""
    local_path = Path(local_path)
    volume_path = Path(volume_path)
    if not local_path.is_file() or local_path.stat().st_size < 1:
        raise OSError(f"Local artifact is absent or empty: {local_path}")
    volume_path.parent.mkdir(parents=True, exist_ok=True)
    expected_bytes, expected_sha256 = artifact_digest(local_path)
    if volume_path.is_file():
        try:
            if artifact_digest(volume_path) == (expected_bytes, expected_sha256):
                return volume_path
        except OSError:
            pass
    last_error = None
    for attempt in range(1, retries + 1):
        partial_path = volume_path.with_name(
            f".{volume_path.name}.{uuid.uuid4().hex}.partial"
        )
        try:
            with local_path.open("rb") as source_handle, partial_path.open(
                "wb"
            ) as destination_handle:
                shutil.copyfileobj(
                    source_handle,
                    destination_handle,
                    length=8 * 1024 * 1024,
                )
                destination_handle.flush()
                os.fsync(destination_handle.fileno())
            if artifact_digest(partial_path) != (expected_bytes, expected_sha256):
                raise OSError("Incomplete temporary Volume artifact copy")
            os.replace(partial_path, volume_path)
            if artifact_digest(volume_path) != (expected_bytes, expected_sha256):
                raise OSError("Published Volume artifact failed verification")
            return volume_path
        except Exception as error:
            last_error = error
            try:
                partial_path.unlink(missing_ok=True)
            except OSError:
                pass
            if attempt < retries:
                delay = 2 ** (attempt - 1)
                print(
                    f"Artifact publish {attempt}/{retries} failed; retrying "
                    f"in {delay}s: {type(error).__name__}: {error}",
                    flush=True,
                )
                time.sleep(delay)
    raise OSError(f"Could not publish artifact: {volume_path}") from last_error


def consolidate_batch_manifests(manifest_paths, expected_image_keys):
    """Read only completed batch manifests and validate current-cohort identity."""
    frames = [pd.read_parquet(path) for path in manifest_paths]
    if not frames:
        return pd.DataFrame()
    consolidated = pd.concat(frames, ignore_index=True, sort=False)
    if "image_key" not in consolidated.columns:
        raise ValueError("A completed anatomy batch lacks image_key")
    consolidated["image_key"] = consolidated["image_key"].astype(str)
    if consolidated["image_key"].duplicated().any():
        duplicated = int(consolidated["image_key"].duplicated().sum())
        raise ValueError(
            f"Completed anatomy batches contain {duplicated} duplicate image keys"
        )
    unexpected = set(consolidated["image_key"]) - set(expected_image_keys)
    if unexpected:
        raise ValueError(
            "Completed anatomy batches contain images outside the current "
            f"cohort ({len(unexpected)} unexpected keys)"
        )
    return consolidated


def publish_segmentation_checkpoint(
    manifest_paths,
    expected_image_keys,
    *,
    total_images,
    total_batches,
    complete,
):
    """Publish a consolidated Parquet checkpoint and an aggregate progress file."""
    checkpoint_frame = consolidate_batch_manifests(
        manifest_paths,
        expected_image_keys,
    )
    local_checkpoint = (
        local_artifact_root
        / f"anatomic_checkpoint_{os.getpid()}_{uuid.uuid4().hex}.parquet"
    )
    volume_checkpoint = (
        checkpoint_root / "anatomic_explainability_checkpoint.parquet"
    )
    write_frame(checkpoint_frame, local_checkpoint)
    publish_local_artifact(local_checkpoint, volume_checkpoint)
    local_checkpoint.unlink(missing_ok=True)

    state = {
        "completed_images": int(len(checkpoint_frame)),
        "total_images": int(total_images),
        "completed_batches": int(len(manifest_paths)),
        "total_batches": int(total_batches),
        "complete": bool(complete),
        "checkpoint_path": str(volume_checkpoint),
        "updated_unix_seconds": float(time.time()),
    }
    local_state = (
        local_artifact_root
        / f"segmentation_progress_{os.getpid()}_{uuid.uuid4().hex}.json"
    )
    volume_state = checkpoint_root / "segmentation_progress.json"
    write_json(state, local_state)
    publish_local_artifact(local_state, volume_state)
    local_state.unlink(missing_ok=True)
    print(
        "Checkpoint published: "
        f"{len(checkpoint_frame):,}/{total_images:,} images across "
        f"{len(manifest_paths):,}/{total_batches:,} batches",
        flush=True,
    )
    return checkpoint_frame, state


def release_batch_memory():
    """Release Python, plotting, Torch, and glibc caches between model batches."""
    plt.close("all")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    try:
        import ctypes

        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass


def sigmoid(value):
    return float(1.0 / (1.0 + np.exp(-np.clip(value, -40, 40))))


def occlusion_metrics(record, masks, retina):
    head = fold_heads[int(record["fold"])]
    model_array = prepare_model_input(record["image_path"], quality_config)
    shape = model_array.shape[:2]
    retina_model = resize_mask(retina, shape)

    def score(mask=None):
        altered = model_array.copy()
        if mask is not None:
            altered[mask] = 0.0
        return linear_head_score_from_array(
            retfound_model,
            head["coefficients"],
            head["intercept"],
            altered,
            retfound_device,
        )

    baseline = score()
    stored_logit = float(record["classifier_logit_oof"])
    replay_error = abs(baseline - stored_logit)
    if replay_error > 1e-3:
        raise RuntimeError(
            "CLSA targeted occlusion did not reproduce the held-out stored "
            f"logit within 0.001: difference={replay_error}"
        )
    output = {
        "occlusion_baseline_logit": baseline,
        "occlusion_baseline_probability": sigmoid(baseline),
        "occlusion_stored_logit_replay_error": replay_error,
    }
    localized = (
        masks["optic_disc_plus_peripapillary"] | masks["fovea_roi"]
    )
    excluded_controls = localized | masks["vessels"]
    for region_name in (
        "optic_disc_roi",
        "optic_disc_plus_peripapillary",
        "fovea_roi",
        "vessels_elsewhere",
    ):
        source_mask = masks[region_name]
        if region_name == "vessels_elsewhere" and vessel_dilation_px:
            source_mask = ndimage.binary_dilation(
                source_mask,
                iterations=vessel_dilation_px,
            ) & retina
        region = resize_mask(source_mask, shape) & retina_model
        if not region.any():
            continue
        region_drop = baseline - score(region)
        if region_name == "vessels_elsewhere":
            controls = sample_translated_control_masks(
                region,
                retina_model,
                resize_mask(excluded_controls, shape),
                n_masks=occlusion_controls,
                random_state=20260815 + int(record.name),
            )
        else:
            controls = sample_equal_area_control_masks(
                retina_model,
                resize_mask(excluded_controls, shape),
                target_area=int(region.sum()),
                n_masks=occlusion_controls,
                random_state=20260815 + int(record.name),
            )
        control_drops = np.asarray(
            [baseline - score(control) for control in controls],
            dtype=float,
        )
        output[f"{region_name}_occlusion_logit_drop"] = float(region_drop)
        output[f"{region_name}_control_drop_median"] = (
            float(np.median(control_drops)) if len(control_drops) else np.nan
        )
        output[f"{region_name}_specific_occlusion_drop"] = (
            float(region_drop - np.median(control_drops))
            if len(control_drops)
            else np.nan
        )
        output[f"{region_name}_n_control_masks"] = int(len(control_drops))
    return output


def save_overlay(rgb, attribution_grid, masks, label, output_path):
    resized = ndimage.zoom(
        np.asarray(attribution_grid, dtype=float),
        (
            rgb.shape[0] / attribution_grid.shape[0],
            rgb.shape[1] / attribution_grid.shape[1],
        ),
        order=1,
    )[: rgb.shape[0], : rgb.shape[1]]
    limit = max(float(np.quantile(np.abs(resized), 0.99)), 1e-8)
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(rgb)
    axes[0].set_title("Processed CLSA fundus")
    axes[1].imshow(rgb)
    axes[1].imshow(masks["vessels"], cmap="Reds", alpha=0.6)
    axes[1].set_title("FR-U-Net vessels")
    axes[2].imshow(resized, cmap="coolwarm", vmin=-limit, vmax=limit)
    axes[2].set_title("Exact glaucoma contribution")
    axes[3].imshow(rgb)
    axes[3].imshow(
        resized,
        cmap="coolwarm",
        alpha=0.45,
        vmin=-limit,
        vmax=limit,
    )
    for name, color in (
        ("optic_disc_roi", "cyan"),
        ("optic_disc_plus_peripapillary", "yellow"),
        ("fovea_roi", "lime"),
        ("vessels", "red"),
    ):
        axes[3].contour(
            masks[name], levels=[0.5], colors=[color], linewidths=0.8
        )
    axes[3].set_title(f"Anatomic overlay | label={int(label)}")
    for axis in axes:
        axis.axis("off")
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


## 4. Resumable CLSA anatomy inference


In [ ]:
ordered = clsa_images.reset_index(drop=True)
expected_image_keys = set(ordered["image_key"].astype(str))
completed_manifest_paths = []
n_batches = math.ceil(len(ordered) / segmentation_batch_size)
expected_manifest_paths = [
    batch_root
    / f"batch_{start:07d}_{min(start + segmentation_batch_size, len(ordered)):07d}"
    / "anatomic_explainability.parquet"
    for start in range(0, len(ordered), segmentation_batch_size)
]
completed_images = 0
new_batches_this_run = 0
completed_run_loaded = False
image_anatomy = None
pipeline_started = time.perf_counter()
print(
    f"Anatomy plan: {len(ordered):,} images in {n_batches:,} durable batches "
    f"of at most {segmentation_batch_size}; this kernel will process at most "
    f"{max_new_batches_per_run or 'all'} new batches",
    flush=True,
)

# Completed runs take a single fast path. Presence alone is not trusted: the
# batch Parquets must consolidate to exactly one row for every current image
# key and retain the columns required by downstream inference. Masks and
# overlays are checked later only where those artifacts are actually consumed.
all_batch_parquets_present = resume_segmentation and all(
    path.is_file() and path.stat().st_size > 0
    for path in expected_manifest_paths
)
if all_batch_parquets_present:
    completed_candidate = consolidate_batch_manifests(
        expected_manifest_paths,
        expected_image_keys,
    )
    required_completed_columns = {
        "image_key",
        "mask_path",
        "overlay_path",
        "fit_source_commit",
        "anatomy_valid",
        "vessels_positive_enrichment",
        "optic_disc_roi_positive_enrichment",
        "fovea_roi_positive_enrichment",
    }
    if run_targeted_occlusion:
        required_completed_columns.add(
            "optic_disc_roi_specific_occlusion_drop"
        )
    missing_completed_columns = required_completed_columns - set(
        completed_candidate.columns
    )
    completed_keys = set(completed_candidate["image_key"].astype(str))
    if missing_completed_columns:
        raise ValueError(
            "All anatomy batch Parquets are present, but the completed run is "
            f"missing columns: {sorted(missing_completed_columns)}"
        )
    if (
        len(completed_candidate) != len(ordered)
        or completed_keys != expected_image_keys
    ):
        raise ValueError(
            "All anatomy batch Parquets are present, but they do not exactly "
            "cover the current CLSA image cohort. No anatomy was recomputed."
        )
    image_anatomy = completed_candidate
    completed_manifest_paths = list(expected_manifest_paths)
    completed_images = len(image_anatomy)
    completed_run_loaded = True
    print(
        "All expected anatomy batch Parquets are present and valid: loaded "
        f"{completed_images:,} rows from {n_batches:,} batches. Skipping "
        "anatomy inference and checkpoint republishing.",
        flush=True,
    )

for batch_index, start in enumerate(
    []
    if completed_run_loaded
    else range(0, len(ordered), segmentation_batch_size),
    start=1,
):
    stop = min(start + segmentation_batch_size, len(ordered))
    batch = ordered.iloc[start:stop].copy()
    batch_dir = batch_root / f"batch_{start:07d}_{stop:07d}"
    batch_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = batch_dir / "anatomic_explainability.parquet"
    if resume_segmentation and manifest_path.exists():
        existing = pd.read_parquet(manifest_path)
        required_resume = {
            "image_key",
            "mask_path",
            "overlay_path",
            "fit_source_commit",
            "anatomy_valid",
            "vessels_positive_enrichment",
            "optic_disc_roi_positive_enrichment",
            "fovea_roi_positive_enrichment",
        }
        occlusion_ready = (
            not run_targeted_occlusion
            or "optic_disc_roi_specific_occlusion_drop" in existing.columns
        )
        input_matches = (
            "image_key" in existing.columns
            and set(existing["image_key"].astype(str))
            == set(batch["image_key"].astype(str))
        )
        artifact_paths = []
        if {"mask_path", "overlay_path"}.issubset(existing.columns):
            artifact_paths = [
                Path(value)
                for column in ("mask_path", "overlay_path")
                for value in existing[column].dropna().astype(str)
            ]
        artifacts_ready = bool(artifact_paths) and all(
            path.is_file() and path.stat().st_size > 0
            for path in artifact_paths
        )
        if (
            required_resume.issubset(existing.columns)
            and occlusion_ready
            and input_matches
            and artifacts_ready
        ):
            completed_images += len(existing)
            print(
                f"[anatomy {batch_index}/{n_batches}] resumed "
                f"{len(existing)} rows; progress={completed_images:,}/"
                f"{len(ordered):,}",
                flush=True,
            )
            completed_manifest_paths.append(manifest_path)
            del existing, batch
            continue
    batch_started = time.perf_counter()
    print(
        f"[anatomy {batch_index}/{n_batches}] images {start:,}:{stop:,} | "
        "stage=preprocess",
        flush=True,
    )
    processed_images = []
    for record in batch.itertuples():
        processed = preprocess_fundus(record.image_path, quality_config)
        processed_images.append(np.asarray(processed.image.convert("RGB")))
    stage_started = time.perf_counter()
    print(
        f"[anatomy {batch_index}/{n_batches}] stage=landmarks",
        flush=True,
    )
    coordinates = landmark_model.predict(processed_images)
    coordinates = np.asarray(coordinates, dtype=float).reshape(-1, 4)
    print(
        f"[anatomy {batch_index}/{n_batches}] landmarks complete in "
        f"{time.perf_counter() - stage_started:.1f}s | stage=vessels",
        flush=True,
    )
    stage_started = time.perf_counter()
    vessel_predictions = fit.ensemble_predict_segmentation(
        vessel_ensemble,
        processed_images,
        device=device,
        size=(512, 512),
        threshold=vessel_threshold,
    )
    vessel_predictions = np.asarray(vessel_predictions, dtype=float)
    if vessel_predictions.ndim == 2:
        vessel_predictions = vessel_predictions[None, ...]
    print(
        f"[anatomy {batch_index}/{n_batches}] vessels complete in "
        f"{time.perf_counter() - stage_started:.1f}s | stage=artifacts",
        flush=True,
    )
    rows = []
    for local_position, (_, record) in enumerate(batch.iterrows()):
        rgb = processed_images[local_position]
        vessel_mask = vessel_predictions[local_position]
        if vessel_mask.shape != rgb.shape[:2]:
            vessel_mask = np.asarray(
                Image.fromarray(vessel_mask.astype(np.float32)).resize(
                    (rgb.shape[1], rgb.shape[0]),
                    getattr(Image.Resampling, "BILINEAR", Image.BILINEAR),
                )
            )
        proxies = fundus_physiology_proxies(rgb)
        masks, anatomy_metadata = build_anatomic_masks(
            rgb.shape[:2],
            coordinates[local_position],
            vessel_mask,
            proxies["retina"],
            optic_disc_radius_scale=optic_disc_radius_scale,
            fovea_radius_scale=fovea_radius_scale,
            peripapillary_multiplier=peripapillary_multiplier,
        )
        with np.load(record["attribution_npz_path"]) as saved_map:
            attribution_grid = np.asarray(
                saved_map["variable_grid"], dtype=float
            )
        region_metrics = attribution_region_metrics(
            attribution_grid,
            proxies["retina"],
            masks,
        )
        key = stable_key(record["image_path"])
        mask_path = mask_root / f"{key}_anatomic_masks.npz"
        local_mask_path = local_artifact_root / mask_path.name
        np.savez_compressed(
            local_mask_path,
            **{name: value.astype(np.uint8) for name, value in masks.items()},
        )
        with np.load(local_mask_path) as saved_masks:
            if set(masks) != set(saved_masks.files):
                raise OSError(f"Staged anatomy mask archive is invalid: {local_mask_path}")
        publish_local_artifact(local_mask_path, mask_path)
        overlay_path = overlay_root / f"{key}_anatomic_overlay.png"
        local_overlay_path = local_artifact_root / overlay_path.name
        save_overlay(
            rgb,
            attribution_grid,
            masks,
            record["glaucoma_label"],
            local_overlay_path,
        )
        publish_local_artifact(local_overlay_path, overlay_path)
        local_mask_path.unlink(missing_ok=True)
        local_overlay_path.unlink(missing_ok=True)
        occlusion = (
            occlusion_metrics(record, masks, proxies["retina"])
            if run_targeted_occlusion and anatomy_metadata["anatomy_valid"]
            else {}
        )
        rows.append(
            {
                "image_key": str(record["image_key"]),
                "artifact_key": key,
                "image_path": record["image_path"],
                "participant_id": str(record["participant_id"]),
                "visit": record.get("visit"),
                "eye": record.get("eye"),
                "glaucoma_label": int(record["glaucoma_label"]),
                "fold": int(record["fold"]),
                "fit_source_commit": FIT_SOURCE_COMMIT,
                "fit_version": fit.__version__,
                "mask_path": str(mask_path),
                "overlay_path": str(overlay_path),
                **anatomy_metadata,
                **region_metrics,
                **occlusion,
            }
        )
        print(
            f"[anatomy {batch_index}/{n_batches}] artifact "
            f"{local_position + 1}/{len(batch)} ready; "
            f"anatomy_valid={anatomy_metadata['anatomy_valid']}",
            flush=True,
        )
    batch_manifest = pd.DataFrame(rows)
    local_manifest_path = (
        local_artifact_root
        / f"batch_{start:07d}_{stop:07d}_anatomic_explainability.parquet"
    )
    write_frame(batch_manifest, local_manifest_path)
    publish_local_artifact(local_manifest_path, manifest_path)
    local_manifest_path.unlink(missing_ok=True)
    completed_manifest_paths.append(manifest_path)
    completed_images += len(batch_manifest)
    new_batches_this_run += 1
    print(
        f"[anatomy {batch_index}/{n_batches}] saved {len(batch_manifest)} rows; "
        f"valid={int(batch_manifest['anatomy_valid'].sum())}; "
        f"progress={completed_images:,}/{len(ordered):,}; "
        f"batch_seconds={time.perf_counter() - batch_started:.1f}",
        flush=True,
    )

    del (
        processed_images,
        coordinates,
        vessel_predictions,
        rows,
        batch_manifest,
        batch,
        rgb,
        vessel_mask,
        proxies,
        masks,
        anatomy_metadata,
        attribution_grid,
        region_metrics,
        occlusion,
    )
    release_batch_memory()

    if new_batches_this_run % checkpoint_every_batches == 0:
        checkpoint_frame, checkpoint_state = publish_segmentation_checkpoint(
            completed_manifest_paths,
            expected_image_keys,
            total_images=len(ordered),
            total_batches=n_batches,
            complete=len(completed_manifest_paths) == n_batches,
        )
        del checkpoint_frame, checkpoint_state
        release_batch_memory()

    if (
        max_new_batches_per_run > 0
        and new_batches_this_run >= max_new_batches_per_run
        and len(completed_manifest_paths) < n_batches
    ):
        print(
            "Reached max_new_batches_per_run; publishing a clean restart "
            "checkpoint before stopping.",
            flush=True,
        )
        break

run_complete = completed_run_loaded or len(completed_manifest_paths) == n_batches
if completed_run_loaded:
    checkpoint_state = {
        "completed_images": int(len(image_anatomy)),
        "total_images": int(len(ordered)),
        "completed_batches": int(len(completed_manifest_paths)),
        "total_batches": int(n_batches),
        "complete": True,
        "checkpoint_path": str(
            checkpoint_root / "anatomic_explainability_checkpoint.parquet"
        ),
        "status": "loaded_from_complete_batch_parquets",
    }
else:
    image_anatomy, checkpoint_state = publish_segmentation_checkpoint(
        completed_manifest_paths,
        expected_image_keys,
        total_images=len(ordered),
        total_batches=n_batches,
        complete=run_complete,
    )
print(
    f"Anatomy run ended after {(time.perf_counter() - pipeline_started) / 60:.1f} "
    f"min; complete={run_complete}",
    flush=True,
)
if not run_complete:
    dbutils.notebook.exit(
        json.dumps(
            {
                **checkpoint_state,
                "status": "checkpointed_incomplete",
                "next_action": (
                    "Rerun notebook 09 with resume_segmentation=true; completed "
                    "batches will be skipped."
                ),
            },
            indent=2,
        )
    )

consolidated_anatomy_path = (
    segmentation_root / "clsa_anatomic_explainability_private.parquet"
)
consolidated_is_current = False
if consolidated_anatomy_path.is_file():
    try:
        saved_keys = pd.read_parquet(
            consolidated_anatomy_path,
            columns=["image_key"],
        )["image_key"].astype(str)
        consolidated_is_current = (
            len(saved_keys) == len(ordered)
            and not saved_keys.duplicated().any()
            and set(saved_keys) == expected_image_keys
        )
    except Exception:
        consolidated_is_current = False
if completed_run_loaded and consolidated_is_current:
    print(
        "Consolidated anatomy Parquet already covers the current cohort; "
        "leaving it unchanged and continuing to downstream analysis.",
        flush=True,
    )
else:
    local_consolidated_path = (
        local_artifact_root
        / f"clsa_anatomic_explainability_{os.getpid()}_{uuid.uuid4().hex}.parquet"
    )
    write_frame(image_anatomy, local_consolidated_path)
    publish_local_artifact(local_consolidated_path, consolidated_anatomy_path)
    local_consolidated_path.unlink(missing_ok=True)
display(
    image_anatomy.groupby("glaucoma_label").agg(
        images=("image_path", "count"),
        participants=("participant_id", "nunique"),
        anatomy_valid_fraction=("anatomy_valid", "mean"),
        mean_vessel_fraction=("vessel_fraction_of_retina", "mean"),
    )
)


## 5. Participant-level CLSA glaucoma-versus-healthy inference


In [ ]:
valid_images = image_anatomy[image_anatomy["anatomy_valid"]].copy()
if valid_images.empty:
    raise ValueError("No images passed anatomic localization/segmentation QC")
primary_metrics = [
    "optic_disc_roi_positive_enrichment",
    "peripapillary_annulus_positive_enrichment",
    "optic_disc_plus_peripapillary_positive_enrichment",
    "fovea_roi_positive_enrichment",
    "vessels_positive_enrichment",
    "vessels_elsewhere_positive_enrichment",
    "optic_disc_roi_absolute_enrichment",
    "peripapillary_annulus_absolute_enrichment",
    "fovea_roi_absolute_enrichment",
    "vessels_absolute_enrichment",
]
if run_targeted_occlusion:
    primary_metrics.extend(
        [
            "optic_disc_roi_specific_occlusion_drop",
            "optic_disc_plus_peripapillary_specific_occlusion_drop",
            "fovea_roi_specific_occlusion_drop",
            "vessels_elsewhere_specific_occlusion_drop",
        ]
    )
available_metrics = [
    column for column in primary_metrics if column in valid_images.columns
]
participant_anatomy = valid_images.groupby(
    ["participant_id", "glaucoma_label"], as_index=False
)[available_metrics].mean()
complete_metrics = [
    column
    for column in available_metrics
    if np.isfinite(
        pd.to_numeric(participant_anatomy[column], errors="coerce")
    ).all()
]
participant_anatomy = participant_anatomy[
    ["participant_id", "glaucoma_label", *complete_metrics]
]
if not complete_metrics:
    raise ValueError("No complete participant-level anatomic metrics remain")
if set(participant_anatomy["glaucoma_label"].astype(int)) != {0, 1}:
    raise ValueError("Valid anatomy participants do not include both CLSA groups")
group_counts = participant_anatomy.groupby("glaucoma_label")[
    "participant_id"
].nunique()
if group_counts.min() < 5:
    raise ValueError(
        "At least five valid participants per CLSA group are required for inference"
    )
write_frame(
    participant_anatomy,
    statistics_root / "participant_anatomic_metrics_private.parquet",
)
inference = participant_permutation_inference(
    participant_anatomy,
    complete_metrics,
    permutations=permutations,
    bootstrap_repetitions=bootstrap_repetitions,
)
write_frame(
    inference,
    statistics_root / "clsa_glaucoma_vs_healthy_anatomic_inference.csv",
)
display(inference.sort_values("permutation_p_max_t").round(5))


## 6. Publication figures and interpretation contract


In [ ]:
positive_metrics = [
    "optic_disc_roi_positive_enrichment",
    "peripapillary_annulus_positive_enrichment",
    "optic_disc_plus_peripapillary_positive_enrichment",
    "fovea_roi_positive_enrichment",
    "vessels_elsewhere_positive_enrichment",
]
plot_metrics = [metric for metric in positive_metrics if metric in complete_metrics]
if plot_metrics:
    long_frame = participant_anatomy.melt(
        id_vars=["participant_id", "glaucoma_label"],
        value_vars=plot_metrics,
        var_name="region_metric",
        value_name="positive_attribution_enrichment",
    )
    fig, axis = plt.subplots(figsize=(12, 6))
    groups = []
    labels = []
    for metric in plot_metrics:
        for label in (0, 1):
            groups.append(
                long_frame[
                    (long_frame["region_metric"] == metric)
                    & (long_frame["glaucoma_label"] == label)
                ]["positive_attribution_enrichment"].to_numpy()
            )
            labels.append(f"{metric.replace('_positive_enrichment', '')}\n{['healthy', 'glaucoma'][label]}")
    axis.boxplot(groups, labels=labels, showfliers=False)
    axis.axhline(1.0, color="black", linestyle="--", linewidth=1)
    axis.set_ylabel("Positive attribution enrichment over retinal area")
    axis.set_title("CLSA held-out glaucoma-map evidence by retinal anatomy")
    axis.tick_params(axis="x", rotation=35)
    fig.tight_layout()
    fig.savefig(
        figure_root / "clsa_anatomic_attribution_enrichment.png",
        dpi=200,
    )
    display(fig)
    plt.close(fig)

valid_fraction = float(image_anatomy["anatomy_valid"].mean())
significant_regions = inference.loc[
    inference["permutation_p_max_t"] < 0.05,
    "metric",
].tolist()
summary = {
    "analysis": "CLSA_glaucoma_vs_healthy_anatomic_explainability",
    "scope": "CLSA only; Zeiss excluded",
    "fundus_image_toolbox": {
        "source_commit": FIT_SOURCE_COMMIT,
        "installed_version": fit.__version__,
        "weight_inventory": str(
            output_root / "FIT_MODEL_WEIGHT_INVENTORY.json"
        ),
        "license": "MIT",
        "vessel_model": "FR-U-Net ensemble trained on FIVES",
        "landmark_model": (
            "multitask EfficientNet trained on ADAM, REFUGE, and IDRID"
        ),
    },
    "anatomy_definitions": {
        "vasculature": "pixel segmentation from the FR-U-Net ensemble",
        "optic_disc": "circular ROI around localized center; not segmentation",
        "peripapillary": "annulus around the localized optic-disc ROI",
        "fovea": "circular ROI around localized center; not segmentation",
    },
    "n_images": int(len(image_anatomy)),
    "n_participants": int(image_anatomy["participant_id"].nunique()),
    "anatomy_valid_fraction": valid_fraction,
    "targeted_occlusion_ran": run_targeted_occlusion,
    "max_t_significant_metrics": significant_regions,
    "claim_gate": {
        "participant_level_inference": True,
        "held_out_classifier_maps": True,
        "anatomy_valid_fraction_at_least_0_90": valid_fraction >= 0.90,
        "multiplicity_control": "max-|T| permutation across region metrics",
        "localized_rois_not_mislabeled_as_segmentations": True,
    },
    "limitations": [
        "CLSA glaucoma status is released physician-diagnosed self-report.",
        "The landmark model supplies centers, not optic-disc/fovea boundaries.",
        "External anatomy models require CLSA-specific visual validation.",
        "Vessel masks do not distinguish arteries from veins.",
        "Attribution enrichment is associative; targeted occlusion is stronger but still model-based.",
    ],
    "outputs": {
        "image_anatomy": str(
            segmentation_root / "clsa_anatomic_explainability_private.parquet"
        ),
        "participant_metrics": str(
            statistics_root / "participant_anatomic_metrics_private.parquet"
        ),
        "inference": str(
            statistics_root / "clsa_glaucoma_vs_healthy_anatomic_inference.csv"
        ),
    },
}
write_json(summary, output_root / "CLSA_ANATOMIC_EXPLAINABILITY_SUMMARY.json")
print(json.dumps(summary, indent=2, default=str))


In [ ]:
os.environ.pop("HF_TOKEN", None)
try:
    dbutils.widgets.remove("hf_token")
except Exception:
    pass
print("Notebook 09 complete; temporary Hugging Face token widget removed")


## 7. Anatomy and attribution review figures

These cells reuse the completed notebook 08 attribution maps and the
completed notebook 09 anatomy artifacts. They do **not** rerun RETFound
or the Fundus Image Toolbox models.

For group averages, every image is affinely registered to a common
disc--fovea axis (disc left, fovea right) before averaging. This
canonicalizes laterality, rotation, and disc--fovea scale. It is an
visualization registration—not a new disease-model input.

The confidence-tail comparison is an **exploratory post hoc analysis**.
It uses participant-held-out glaucoma probabilities and compares the
highest 10% (glaucoma-like) with the lowest 10% (healthy-like) among all
matched participants with completed notebook 08/09 artifacts. It must
not be interpreted as independent validation.


In [ ]:
from pathlib import Path
import os
import sys
import time
import uuid

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage
from scipy.stats import fisher_exact


review_repo_root = Path(
    "/Workspace/Users/ad0038@pennmedicine.upenn.edu/CLSA/CLSA_retina"
)
review_age_glaucoma_root = Path(
    "/Volumes/ophthalmology_analytics/dev_optic/clsa_dataset/derived/"
    "clsa_retinal_aging/Age_Glaucoma"
)
notebook08_root = (
    review_age_glaucoma_root
    / "13_glaucoma_classifier_spatial_validation"
)
output_root = (
    review_age_glaucoma_root / "14_clsa_anatomic_explainability"
)
review_module_root = review_repo_root / "src"
if str(review_module_root) not in sys.path:
    sys.path.insert(0, str(review_module_root))

from clsa_anatomic_explainability import (  # noqa: E402
    disc_fovea_affine_matrix,
    participant_permutation_inference,
    select_probability_extremes,
)
from fundus_retfound_pipeline import (  # noqa: E402
    QualityConfig,
    preprocess_fundus,
    write_frame,
    write_json,
)

review_root = output_root / "05_review_figures"
review_root.mkdir(parents=True, exist_ok=True)
segmentation_root = output_root / "01_image_anatomy"
statistics_root = output_root / "03_statistics"
attribution_batch_root = notebook08_root / "03_patch_attributions" / "batches"
review_local_artifact_root = Path(
    f"/local_disk0/tmp/clsa_anatomic_review_{os.getpid()}"
)
review_local_artifact_root.mkdir(parents=True, exist_ok=True)
quality_config = QualityConfig(
    output_size=256,
    model_input_size=224,
    save_preprocessed=False,
)
permutations = 5000
bootstrap_repetitions = 2000
review_examples_per_group = 3
confidence_extreme_fraction = 0.10
registration_size = 256


def publish_review_artifact(local_path, volume_path, retries=4):
    """Use the same verified Volume publisher as anatomy checkpoints."""
    return publish_local_artifact(local_path, volume_path, retries=retries)

saved_image_anatomy_path = (
    segmentation_root / "clsa_anatomic_explainability_private.parquet"
)
saved_participant_anatomy_path = (
    statistics_root / "participant_anatomic_metrics_private.parquet"
)
participant_prediction_path = (
    notebook08_root
    / "01_participant_classifier"
    / "CLSA_glaucoma_participant_oof_predictions.parquet"
)
for label, path in {
    "image anatomy": saved_image_anatomy_path,
    "participant anatomy": saved_participant_anatomy_path,
    "participant predictions": participant_prediction_path,
}.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {label} output: {path}")

review_images_all = pd.read_parquet(saved_image_anatomy_path)
review_images_all["participant_id"] = review_images_all[
    "participant_id"
].astype(str)
review_images = review_images_all[review_images_all["anatomy_valid"]].copy()
participant_anatomy_review = pd.read_parquet(saved_participant_anatomy_path)
participant_anatomy_review["participant_id"] = (
    participant_anatomy_review["participant_id"].astype(str)
)
participant_predictions_review = pd.read_parquet(participant_prediction_path)
participant_predictions_review["participant_id"] = (
    participant_predictions_review["participant_id"].astype(str)
)
prediction_columns = [
    "participant_id",
    "glaucoma_label",
    "glaucoma_probability_oof",
    "classifier_logit_oof",
]
missing_prediction_columns = set(prediction_columns) - set(
    participant_predictions_review.columns
)
if missing_prediction_columns:
    raise ValueError(
        "Notebook 08 participant predictions are missing: "
        f"{sorted(missing_prediction_columns)}"
    )
participant_predictions_review = participant_predictions_review[
    prediction_columns
].drop_duplicates("participant_id")

expected_matched_participants = int(
    participant_predictions_review["participant_id"].nunique()
)
anatomy_artifact_participants = int(
    review_images_all["participant_id"].nunique()
)
print(
    "Full matched-cohort anatomy coverage: "
    f"{anatomy_artifact_participants:,}/{expected_matched_participants:,} "
    "participants",
    flush=True,
)
if anatomy_artifact_participants != expected_matched_participants:
    raise RuntimeError(
        "Notebook 09 does not yet contain all matched CLSA participants: "
        f"{anatomy_artifact_participants:,} completed versus "
        f"{expected_matched_participants:,} expected. Rerun notebook 08 with "
        "explain_all_matched_clsa=true and "
        "include_zeiss_in_explainability=false, then rerun notebook 09 with "
        "maximum_images=0."
    )

review_images = review_images.merge(
    participant_predictions_review,
    on=["participant_id", "glaucoma_label"],
    how="left",
    validate="many_to_one",
)
if review_images["glaucoma_probability_oof"].isna().any():
    raise ValueError("Some anatomy images lack participant-held-out scores")

attribution_map_paths = {
    path.name.replace("_exact_glaucoma_attribution.npz", ""): str(path)
    for path in attribution_batch_root.rglob(
        "*_exact_glaucoma_attribution.npz"
    )
}
review_images["attribution_npz_path"] = review_images["image_key"].map(
    attribution_map_paths
)
if review_images["attribution_npz_path"].isna().any():
    raise FileNotFoundError(
        "One or more valid anatomy images lack an attribution-map artifact"
    )

score_anatomy = participant_anatomy_review.merge(
    participant_predictions_review,
    on=["participant_id", "glaucoma_label"],
    how="inner",
    validate="one_to_one",
)
confidence_extremes = select_probability_extremes(
    score_anatomy,
    fraction=confidence_extreme_fraction,
)
extreme_lookup = confidence_extremes.set_index("participant_id")[
    "confidence_extreme"
].to_dict()
review_images["confidence_extreme"] = review_images["participant_id"].map(
    extreme_lookup
)

confidence_audit = (
    confidence_extremes.groupby("confidence_extreme", as_index=False)
    .agg(
        participants=("participant_id", "nunique"),
        known_glaucoma_fraction=("glaucoma_label", "mean"),
        mean_held_out_probability=("glaucoma_probability_oof", "mean"),
        minimum_held_out_probability=("glaucoma_probability_oof", "min"),
        maximum_held_out_probability=("glaucoma_probability_oof", "max"),
    )
    .sort_values("mean_held_out_probability")
)
display(confidence_audit.round(4))


In [ ]:
def load_registered_review_arrays(record, output_size=256):
    """Load one completed case and register it to the disc--fovea axis."""
    processed = preprocess_fundus(record["image_path"], quality_config)
    rgb = np.asarray(processed.image.convert("RGB"), dtype=np.uint8)
    with np.load(record["mask_path"]) as saved_masks:
        masks = {
            name: np.asarray(saved_masks[name], dtype=bool)
            for name in saved_masks.files
        }
    with np.load(record["attribution_npz_path"]) as saved_attribution:
        attribution_grid = np.asarray(
            saved_attribution["variable_grid"], dtype=float
        )
    attribution = ndimage.zoom(
        attribution_grid,
        (
            rgb.shape[0] / attribution_grid.shape[0],
            rgb.shape[1] / attribution_grid.shape[1],
        ),
        order=1,
    )[: rgb.shape[0], : rgb.shape[1]]
    retina = np.logical_or.reduce(list(masks.values()))
    attribution_scale = max(
        float(np.quantile(np.abs(attribution[retina]), 0.99)), 1e-8
    )
    attribution = np.clip(attribution / attribution_scale, -1.0, 1.0)
    matrix = disc_fovea_affine_matrix(
        (
            record["fovea_x_px"],
            record["fovea_y_px"],
            record["optic_disc_x_px"],
            record["optic_disc_y_px"],
        ),
        output_size=output_size,
    )

    def warp(array, interpolation, border_value=0):
        return cv2.warpAffine(
            array,
            matrix,
            (output_size, output_size),
            flags=interpolation,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=border_value,
        )

    registered_masks = {
        name: warp(mask.astype(np.uint8), cv2.INTER_NEAREST) > 0
        for name, mask in masks.items()
    }
    return {
        "rgb": warp(rgb, cv2.INTER_LINEAR),
        "attribution": warp(
            attribution.astype(np.float32), cv2.INTER_LINEAR
        ),
        "retina": warp(retina.astype(np.uint8), cv2.INTER_NEAREST) > 0,
        "masks": registered_masks,
    }


def save_review_figure(fig, filename):
    """Stage a complete PNG locally before publishing it to the Volume."""
    local_path = review_local_artifact_root / filename
    destination = review_root / filename
    fig.savefig(local_path, dpi=220, bbox_inches="tight")
    publish_review_artifact(local_path, destination)
    local_path.unlink(missing_ok=True)
    display(fig)
    plt.close(fig)
    return destination


def add_roi_contours(axis, masks, linewidth=1.2):
    for name, color, label in (
        ("optic_disc_roi", "cyan", "optic-disc ROI"),
        ("peripapillary_annulus", "yellow", "peripapillary annulus"),
        ("fovea_roi", "lime", "foveal ROI"),
    ):
        mask = masks[name]
        if mask.any():
            axis.contour(
                mask,
                levels=[0.5],
                colors=[color],
                linewidths=linewidth,
            )
    axis.text(
        0.01,
        0.02,
        "cyan=disc | yellow=peripapillary | green=fovea | red=vessels",
        transform=axis.transAxes,
        fontsize=6,
        color="white",
        bbox={"facecolor": "black", "alpha": 0.65, "pad": 2},
    )


### 7A. Deidentified representative ROI and attribution panels

Within each known CLSA group, cases are selected at evenly spaced score
ranks rather than by the appearance of their heatmaps.


In [ ]:
representatives = []
for glaucoma_label, group in review_images.groupby("glaucoma_label"):
    ordered_group = group.sort_values(
        ["glaucoma_probability_oof", "image_key"], kind="stable"
    ).reset_index(drop=True)
    positions = np.linspace(
        0,
        len(ordered_group) - 1,
        min(review_examples_per_group, len(ordered_group)),
    ).round().astype(int)
    representatives.append(ordered_group.iloc[np.unique(positions)])
representatives = pd.concat(representatives, ignore_index=True)

fig, axes = plt.subplots(
    len(representatives), 4, figsize=(14, 3.2 * len(representatives))
)
if len(representatives) == 1:
    axes = axes[None, :]
for row_index, (_, record) in enumerate(representatives.iterrows()):
    arrays = load_registered_review_arrays(record, registration_size)
    rgb = arrays["rgb"]
    masks = arrays["masks"]
    attribution = arrays["attribution"]
    axes[row_index, 0].imshow(rgb)
    axes[row_index, 0].set_title(
        f"Known {'glaucoma' if int(record['glaucoma_label']) else 'healthy'} | "
        f"held-out p={record['glaucoma_probability_oof']:.2f}"
    )
    axes[row_index, 1].imshow(rgb)
    axes[row_index, 1].imshow(masks["vessels"], cmap="Reds", alpha=0.58)
    add_roi_contours(axes[row_index, 1], masks)
    axes[row_index, 1].set_title("Registered anatomy")
    axes[row_index, 2].imshow(
        attribution, cmap="coolwarm", vmin=-1, vmax=1
    )
    axes[row_index, 2].set_title("Signed attribution (within-image scaled)")
    axes[row_index, 3].imshow(rgb)
    axes[row_index, 3].imshow(
        attribution, cmap="coolwarm", vmin=-1, vmax=1, alpha=0.48
    )
    add_roi_contours(axes[row_index, 3], masks)
    axes[row_index, 3].set_title("Attribution + anatomy")
    for axis in axes[row_index]:
        axis.axis("off")
fig.suptitle(
    "CLSA glaucoma explainability review: anatomy-registered examples",
    fontsize=15,
    y=1.002,
)
fig.tight_layout()
representative_figure_path = save_review_figure(
    fig, "clsa_registered_representative_roi_heatmaps.png"
)


### 7B. Registered group-average images, segmentations, and heatmaps

Vessel panels show the fraction of registered images containing a
vessel at each pixel. ROI panels show the corresponding localization
occupancy. Attribution maps are scaled within image by their 99th
absolute percentile before averaging, so they describe spatial pattern
rather than allowing a few high-magnitude maps to dominate.


In [ ]:
average_group_names = [
    "known_healthy",
    "known_glaucoma",
    "bottom_healthy_like",
    "top_glaucoma_like",
]


def empty_average_accumulator(size):
    return {
        "n_images": 0,
        "rgb_sum": np.zeros((size, size, 3), dtype=np.float64),
        "coverage": np.zeros((size, size), dtype=np.float64),
        "attribution_sum": np.zeros((size, size), dtype=np.float64),
        "vessels_sum": np.zeros((size, size), dtype=np.float64),
        "optic_disc_sum": np.zeros((size, size), dtype=np.float64),
        "peripapillary_sum": np.zeros((size, size), dtype=np.float64),
        "fovea_sum": np.zeros((size, size), dtype=np.float64),
    }


average_accumulators = {
    name: empty_average_accumulator(registration_size)
    for name in average_group_names
}


def update_average(accumulator, arrays):
    retina = arrays["retina"].astype(float)
    masks = arrays["masks"]
    accumulator["n_images"] += 1
    accumulator["coverage"] += retina
    accumulator["rgb_sum"] += (
        arrays["rgb"].astype(float) / 255.0
    ) * retina[..., None]
    accumulator["attribution_sum"] += arrays["attribution"] * retina
    accumulator["vessels_sum"] += masks["vessels"].astype(float)
    accumulator["optic_disc_sum"] += masks["optic_disc_roi"].astype(float)
    accumulator["peripapillary_sum"] += masks[
        "peripapillary_annulus"
    ].astype(float)
    accumulator["fovea_sum"] += masks["fovea_roi"].astype(float)


for image_index, (_, record) in enumerate(review_images.iterrows(), start=1):
    arrays = load_registered_review_arrays(record, registration_size)
    known_group = (
        "known_glaucoma"
        if int(record["glaucoma_label"]) == 1
        else "known_healthy"
    )
    update_average(average_accumulators[known_group], arrays)
    if pd.notna(record["confidence_extreme"]):
        update_average(
            average_accumulators[str(record["confidence_extreme"])], arrays
        )
    if image_index % 25 == 0 or image_index == len(review_images):
        print(
            f"[review averages] {image_index:,}/{len(review_images):,} images",
            flush=True,
        )


def finalize_average(accumulator):
    coverage = np.maximum(accumulator["coverage"], 1.0)
    result = {
        "n_images": int(accumulator["n_images"]),
        "mean_rgb": accumulator["rgb_sum"] / coverage[..., None],
        "mean_signed_attribution": (
            accumulator["attribution_sum"] / coverage
        ),
        "mean_vessel_prevalence": accumulator["vessels_sum"] / coverage,
        "optic_disc_occupancy": accumulator["optic_disc_sum"] / coverage,
        "peripapillary_occupancy": (
            accumulator["peripapillary_sum"] / coverage
        ),
        "fovea_occupancy": accumulator["fovea_sum"] / coverage,
        "retinal_coverage": accumulator["coverage"],
    }
    return result


registered_averages = {
    name: finalize_average(accumulator)
    for name, accumulator in average_accumulators.items()
}
average_cache_arrays = {}
for group_name, values in registered_averages.items():
    for array_name, value in values.items():
        average_cache_arrays[f"{group_name}__{array_name}"] = value
local_average_cache = (
    review_local_artifact_root / "registered_group_averages.npz"
)
np.savez_compressed(local_average_cache, **average_cache_arrays)
registered_average_path = publish_review_artifact(
    local_average_cache,
    review_root / "registered_group_averages.npz",
)
local_average_cache.unlink(missing_ok=True)


def plot_registered_averages(group_names, display_names, filename, title):
    attribution_limit = max(
        float(
            np.quantile(
                np.concatenate(
                    [
                        np.abs(
                            registered_averages[name][
                                "mean_signed_attribution"
                            ]
                        ).ravel()
                        for name in group_names
                    ]
                ),
                0.99,
            )
        ),
        1e-6,
    )
    vessel_limit = max(
        0.25,
        max(
            float(
                np.quantile(
                    registered_averages[name]["mean_vessel_prevalence"],
                    0.995,
                )
            )
            for name in group_names
        ),
    )
    fig, axes = plt.subplots(
        len(group_names), 4, figsize=(14, 3.5 * len(group_names))
    )
    if len(group_names) == 1:
        axes = axes[None, :]
    for row_index, (group_name, display_name) in enumerate(
        zip(group_names, display_names)
    ):
        average = registered_averages[group_name]
        axes[row_index, 0].imshow(np.clip(average["mean_rgb"], 0, 1))
        axes[row_index, 0].set_title(
            f"{display_name}: mean fundus (n={average['n_images']} images)"
        )
        anatomy_rgb = np.zeros(
            (registration_size, registration_size, 3), dtype=float
        )
        anatomy_rgb[..., 0] = np.maximum(
            average["mean_vessel_prevalence"],
            average["peripapillary_occupancy"],
        )
        anatomy_rgb[..., 1] = np.maximum(
            average["peripapillary_occupancy"],
            average["fovea_occupancy"],
        )
        anatomy_rgb[..., 2] = average["optic_disc_occupancy"]
        axes[row_index, 1].imshow(np.clip(anatomy_rgb, 0, 1))
        axes[row_index, 1].set_title("Mean ROI occupancy")
        vessel_image = axes[row_index, 2].imshow(
            average["mean_vessel_prevalence"],
            cmap="magma",
            vmin=0,
            vmax=vessel_limit,
        )
        axes[row_index, 2].set_title("Mean vessel segmentation prevalence")
        fig.colorbar(vessel_image, ax=axes[row_index, 2], fraction=0.046)
        attribution_image = axes[row_index, 3].imshow(
            average["mean_signed_attribution"],
            cmap="coolwarm",
            vmin=-attribution_limit,
            vmax=attribution_limit,
        )
        axes[row_index, 3].set_title("Mean signed RETFound attribution")
        fig.colorbar(
            attribution_image, ax=axes[row_index, 3], fraction=0.046
        )
        for axis in axes[row_index]:
            axis.axis("off")
    fig.suptitle(title, fontsize=15, y=1.01)
    fig.tight_layout()
    return save_review_figure(fig, filename)


known_group_average_figure_path = plot_registered_averages(
    ["known_healthy", "known_glaucoma"],
    ["Known healthy", "Known glaucoma"],
    "clsa_known_group_registered_averages.png",
    "CLSA anatomy-registered group averages",
)
confidence_average_figure_path = plot_registered_averages(
    ["bottom_healthy_like", "top_glaucoma_like"],
    ["Bottom 10%: healthy-like", "Top 10%: glaucoma-like"],
    "clsa_confidence_extremes_registered_averages.png",
    "Exploratory held-out confidence extremes",
)


## 8. Exploratory top-versus-bottom 10% confidence subanalysis

The unit of analysis remains the participant. Because the tails are
defined using the classifier score being interpreted, these estimates
describe how attribution anatomy changes with model confidence; they do
not test generalization or establish a disease mechanism.


In [ ]:
confidence_metric_columns = [
    column
    for column in participant_anatomy_review.columns
    if column not in {"participant_id", "glaucoma_label"}
    and pd.api.types.is_numeric_dtype(participant_anatomy_review[column])
]
confidence_extreme_inference = participant_permutation_inference(
    confidence_extremes,
    confidence_metric_columns,
    label_column="confidence_extreme_code",
    permutations=permutations,
    bootstrap_repetitions=bootstrap_repetitions,
    random_state=20260816,
).rename(
    columns={
        "n_glaucoma": "n_top_glaucoma_like",
        "n_healthy": "n_bottom_healthy_like",
        "glaucoma_minus_healthy": "top_minus_bottom",
    }
)
write_frame(
    confidence_extreme_inference,
    review_root / "confidence_extreme_inference.csv",
)

confidence_label_table = pd.crosstab(
    confidence_extremes["confidence_extreme"],
    confidence_extremes["glaucoma_label"],
).reindex(
    index=["bottom_healthy_like", "top_glaucoma_like"],
    columns=[0, 1],
    fill_value=0,
)
confidence_odds_ratio, confidence_fisher_p = fisher_exact(
    confidence_label_table.to_numpy()
)
display(
    confidence_extreme_inference.sort_values("permutation_p_max_t").round(5)
)
display(
    confidence_audit.assign(
        known_label_fisher_exact_p=confidence_fisher_p,
        known_label_odds_ratio=confidence_odds_ratio,
    ).round(5)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for glaucoma_label, color, label in (
    (0, "#4C78A8", "Known healthy"),
    (1, "#E45756", "Known glaucoma"),
):
    values = score_anatomy.loc[
        score_anatomy["glaucoma_label"] == glaucoma_label,
        "glaucoma_probability_oof",
    ]
    axes[0].hist(
        values,
        bins=np.linspace(0, 1, 16),
        alpha=0.55,
        color=color,
        label=label,
    )
bottom_threshold = float(
    confidence_extremes.loc[
        confidence_extremes["confidence_extreme"] == "bottom_healthy_like",
        "glaucoma_probability_oof",
    ].max()
)
top_threshold = float(
    confidence_extremes.loc[
        confidence_extremes["confidence_extreme"] == "top_glaucoma_like",
        "glaucoma_probability_oof",
    ].min()
)
axes[0].axvline(
    bottom_threshold, color="#4C78A8", linestyle="--", linewidth=2
)
axes[0].axvline(
    top_threshold, color="#E45756", linestyle="--", linewidth=2
)
axes[0].set_xlabel("Participant-held-out glaucoma probability")
axes[0].set_ylabel("Participants")
axes[0].set_title("Explained CLSA participant score distribution")
axes[0].legend(frameon=False)

forest = confidence_extreme_inference.sort_values(
    "top_minus_bottom"
).reset_index(drop=True)
y_positions = np.arange(len(forest))
axes[1].errorbar(
    forest["top_minus_bottom"],
    y_positions,
    xerr=np.vstack(
        [
            np.maximum(
                forest["top_minus_bottom"]
                - forest["bootstrap_95_ci_low"],
                0,
            ),
            np.maximum(
                forest["bootstrap_95_ci_high"]
                - forest["top_minus_bottom"],
                0,
            ),
        ]
    ),
    fmt="o",
    color="#6F4E7C",
    ecolor="#6F4E7C",
    capsize=3,
)
axes[1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_yticks(y_positions)
axes[1].set_yticklabels(
    [
        value.replace("_positive_enrichment", "")
        .replace("_absolute_enrichment", " (absolute)")
        .replace("_", " ")
        for value in forest["metric"]
    ],
    fontsize=8,
)
axes[1].set_xlabel("Top 10% minus bottom 10% enrichment")
axes[1].set_title("Participant-level exploratory anatomy differences")
fig.tight_layout()
confidence_statistics_figure_path = save_review_figure(
    fig, "clsa_confidence_extremes_statistics.png"
)

confidence_summary = {
    "analysis": "CLSA_explainability_confidence_extremes",
    "status": "exploratory_post_hoc",
    "selection_population": (
        "participants independently selected for notebook 08 explainability "
        "with valid notebook 09 anatomy"
    ),
    "confidence_score": "participant-held-out glaucoma probability",
    "tail_fraction": confidence_extreme_fraction,
    "n_available_participants": int(score_anatomy["participant_id"].nunique()),
    "n_expected_matched_participants": expected_matched_participants,
    "n_participants_with_anatomy_artifacts": anatomy_artifact_participants,
    "n_bottom_healthy_like": int(
        (confidence_extremes["confidence_extreme_code"] == 0).sum()
    ),
    "n_top_glaucoma_like": int(
        (confidence_extremes["confidence_extreme_code"] == 1).sum()
    ),
    "bottom_probability_maximum": bottom_threshold,
    "top_probability_minimum": top_threshold,
    "known_label_fisher_exact_p": float(confidence_fisher_p),
    "known_label_odds_ratio": float(confidence_odds_ratio),
    "max_t_significant_metrics": confidence_extreme_inference.loc[
        confidence_extreme_inference["permutation_p_max_t"] < 0.05,
        "metric",
    ].tolist(),
    "registration": (
        "affine disc-fovea axis canonicalization for visualization only"
    ),
    "attribution_scaling": (
        "within-image 99th absolute percentile before group averaging"
    ),
    "privacy": "figures and displayed summaries contain no participant IDs",
    "outputs": {
        "representative_figure": str(representative_figure_path),
        "known_group_averages": str(known_group_average_figure_path),
        "confidence_extreme_averages": str(confidence_average_figure_path),
        "confidence_statistics": str(confidence_statistics_figure_path),
        "registered_average_arrays": str(registered_average_path),
        "confidence_inference": str(
            review_root / "confidence_extreme_inference.csv"
        ),
    },
}
write_json(
    confidence_summary,
    review_root / "CLSA_CONFIDENCE_EXTREMES_SUMMARY.json",
)
print(json.dumps(confidence_summary, indent=2, default=str))
